# Moteur Monte Carlo & Le Biais de Discrétisation

## Test

Discrétisation du mouvement brownien géométrique par le schéma d'Euler-Maruyama

Un ordinateur ne pouvant pas simuler un processus en temps continu, nous approchons le mouvement brownien géométrique (GBM) à l'aide du schéma d'Euler-Maruyama. Cette méthode consiste à remplacer la dynamique continue par des évolutions successives sur des pas de temps discrets $\Delta t= \frac{T}{N}$. À chaque pas, l'évolution du prix dépend d'une composante déterministe (le drift) et d'une composante aléatoire modélisée par une variable gaussienne centrée réduite Z∼N(0,1).

In [53]:
# Paramètres de test
S0 = 100
mu = 0.05
sigma = 0.20
T = 1
N = 252
M = 10000

dt = T/N

On construit 10 000 trajectoires du prix $S_t$ en sachant que $S_{t+Δt}=S_t+\mu S_t \Delta t+\sigma S_t \sqrt{\Delta t}*Z=S_t(1+\mu \Delta t+\sigma \sqrt{\Delta t}*Z)$

In [54]:
# On génère une matrice numpy de taille (252, 10000) contenant des tirages aléatoires Z
import numpy as np
Z = np.random.normal(loc=0, scale=1, size=(N,M))

# on calcul les 10 000 trajectoires à partir de Z
facteur_matrice = 1 + mu*dt + sigma*np.sqrt(dt)*Z
St = S0*np.cumprod(facteur_matrice, axis=0) # on travaille sur l'axe des jours et pas trajectoires (252)
print(St.shape)


(252, 10000)


On regarde si on a bien $\mathbb{E}(S_T)=S_0e^{\mu T}$

In [55]:
valeur_theorique = S0*np.exp(mu*T)
valeur_empirique = np.mean(St[-1,:])    # c'est dans la dernière ligne que l'on a S_T
print(valeur_theorique)
print(valeur_empirique)
print(abs(valeur_empirique - valeur_theorique)/valeur_theorique)

105.12710963760242
105.21705440850761
0.0008555811266499364


La moyenne empirique est proche de la valeur théorique $\mathbb{E}(S_T)=S_0e^{\mu T}$, mais elle n'est pas exactement égale à cause de deux sources d'erreur :
- Erreur d'échantillonage (inhérente au modèle de Monte Carlo) : on ne simule que 10 000 trajectoires et pas une infinité
- Biais de discréditasion dû au schéma d'Euler-Maruyama : on approches un processus continue par des pas de taille $\Delta t=\frac{1}{252}$

## On essaye d'appliquer cela à NVDA et SMIC

In [56]:
import pandas as pd

prices = pd.read_csv("../data/processed/prices_aligned.csv", index_col=0, parse_dates=True)
returns = pd.read_csv("../data/processed/returns_clean.csv", index_col=0, parse_dates=True)

prices_nvda = prices['NVDA']
returns_nvda = returns['NVDA']
prices_smic = prices['0981.HK']
returns_smic = returns['0981.HK']

On calcul $\mu$ et $\sigma$ pour NVDA et SMIC

In [57]:
mu_log_nvda = np.nanmean(returns_nvda)
sigma_quotidien_nvda = np.nanstd(returns_nvda)

mu_log_smic = np.nanmean(returns_smic)
sigma_quotidien_smic = np.nanstd(returns_smic)

On annualise et on corrige ITÔ

In [58]:
sigma_annuel_nvda = sigma_quotidien_nvda * np.sqrt(252)
mu_arithmetique_nvda = (mu_log_nvda * 252) + (sigma_annuel_nvda**2 / 2)

sigma_annuel_smic = sigma_quotidien_smic * np.sqrt(252)
mu_arithmetique_smic = (mu_log_smic * 252) + (sigma_annuel_smic**2 / 2)

On récupère S0

In [59]:
S0_nvda = prices['NVDA'].iloc[-1]
S0_smic = prices['0981.HK'].iloc[-1]

In [60]:
print(f"NVDA - Volatilité Annuelle: {sigma_annuel_nvda*100:.2f}% | Drift Annuel: {mu_arithmetique_nvda*100:.2f}%")
print(f"SMIC - Volatilité Annuelle: {sigma_annuel_smic*100:.2f}% | Drift Annuel: {mu_arithmetique_smic*100:.2f}%")

NVDA - Volatilité Annuelle: 44.46% | Drift Annuel: 54.34%
SMIC - Volatilité Annuelle: 51.68% | Drift Annuel: 34.36%


On fait le test en générant M trajectoires puis en regardant l'erreur avec la valeur théorique

In [63]:
# on met les mêmes paramètres qu'avant mais on le fait surtout pour être plus clair
M = 10000
T = 1
N = 252
dt = T/N

# On génère une matrice numpy de taille (252, 10000) contenant des tirages aléatoires Z
import numpy as np
Z = np.random.normal(loc=0, scale=1, size=(N,M))

# on calcul les 10 000 trajectoires à partir de Z
facteur_matrice_nvda = 1 + mu_arithmetique_nvda*dt + sigma_annuel_nvda*np.sqrt(dt)*Z
St_nvda = S0_nvda*np.cumprod(facteur_matrice_nvda, axis=0)

facteur_matrice_smic = 1 + mu_arithmetique_smic*dt + sigma_annuel_smic*np.sqrt(dt)*Z
St_smic = S0_smic*np.cumprod(facteur_matrice_smic, axis=0)


# on fait la vérification
valeur_theorique_nvda = S0_nvda*np.exp(mu_arithmetique_nvda*T)
valeur_empirique_nvda = np.mean(St_nvda[-1,:])
print(valeur_theorique_nvda)
print(valeur_empirique_nvda)
print(abs(valeur_empirique_nvda - valeur_theorique_nvda)/valeur_theorique_nvda)

valeur_theorique_smic = S0_smic*np.exp(mu_arithmetique_smic*T)
valeur_empirique_smic = np.mean(St_smic[-1,:])
print(valeur_theorique_smic)
print(valeur_empirique_smic)
print(abs(valeur_empirique_smic - valeur_theorique_smic)/valeur_theorique_smic)

359.2676926505513
358.443119179327
0.0022951506302747145
111.2545882207322
110.96302740530193
0.002620663292122464


Le moteur de Monte Carlo est validé mathématiquement sur des données de marchés réelles. On le code donc dans src/stochastic.py et on ajoute un test unitaire dans tests/test_stochastic.py